# Physics diagnostics of the COMSOL reference fields

Check whether the physics implementations agree with real COMSOL fields before they are used as losses or metrics. Every number here is computed from **published reference data only**; the notebook trains nothing, writes nothing and never changes a loss.

- **Steady flow:** the dimensionless training terms (momentum, cross-section flux balance, pressure boundary) of the COMSOL solution per derivative backend, interior crop and precision, with the size of the $\tfrac{2}{3}(\nabla\cdot u)$ trace part of the stress and the pointwise divergence variants removed from the loss; finite differences against spectral derivatives on the same fields; and how strongly each term separates controlled field errors from the reference floor.
- **Transient drying:** water-mass and sorption balances from the stored fields and the COMSOL global series, and which balance terms the Dataset and artifacts do not persist.

Analytic operator checks stay in `tests/domain/test_brinkman_residuals.py` and `tests/domain/test_physics_derivatives.py`; this notebook only evaluates real fields. Run it on a compute-node kernel ([Setup and execution](../docs/01_setup_and_execution.md#notebooks-on-compute-nodes)).

In [ ]:
from IPython.display import Markdown
from IPython.display import display as show

from src import common
from src.analysis import physics_diagnostics as diagnostics

STORAGE_ROOT = common.paths.get_storage_root()
STEADY_DATASET = "lentil-train1-val1-test1+chickpea-train1-val1-test1+kidney-bean-ood2__ref-range__lhs-seed-9300"
TRANSIENT_DATASET = "lentil-train1-val1-test1+chickpea-train1-val1-test1+kidney-bean-ood2__ref-range__lhs-seed-9200"
MAX_CASES: int | None = None  # leading Dataset cases per task; None uses all
MAP_CASE = 0  # case position shown in the residual maps
DECOMPOSITION_CROP = 2  # interior crop of the decomposition figure

steady = diagnostics.steady
transient = diagnostics.transient
plots = diagnostics.plots
scientific = "{:.2e}".format

## Steady flow

### Reference cases and maintained loss settings

All terms use the domain scales from the inputs only: mean inlet pressure $\Delta p$, $G = \Delta p / L_y$ and Darcy velocity $U = K_g G / \mu$ with $K_g$ the geometric-mean permeability. The Brinkman length $\sqrt{K_g/\bar\varepsilon}$ against the grid spacing shows whether wall boundary layers are resolved. The second table lists the physics settings of every maintained physics-informed steady configuration.

In [ ]:
reference = diagnostics.sources.load_steady_reference(STEADY_DATASET, storage_root=STORAGE_ROOT, max_cases=MAX_CASES)
show(steady.flow_regime(reference).style.format(scientific, subset=["brinkman_length_m", "brinkman_length_over_dx"]))
show(steady.maintained_physics_settings())

### Reference residual floor per backend, crop and variant

Terms of the COMSOL solution, i.e. the floor below which a loss cannot distinguish a prediction from the reference:

| Column | Definition |
| --- | --- |
| `momentum` | training term: interior mean of $(R_x^2 + R_y^2)/G^2$, $R = -\nabla p + \nabla\cdot\tau - \mu K^{-1}u$, $\tau = (\mu/\varepsilon)(\nabla u + \nabla u^T - \tfrac{2}{3}(\nabla\cdot u) I)$ as in COMSOL |
| `trace_rms` | interior RMS of the trace part of $\nabla\cdot\tau$, $-\tfrac{2}{3}\nabla((\mu/\varepsilon)\nabla\cdot u)/G$ |
| `pointwise_divergence`, `porous_divergence` | interior mean of $(\nabla\cdot u\, L_y/U)^2$ and $(\nabla\cdot(\varepsilon u)\, L_y/(\bar\varepsilon U))^2$, removed from the loss |
| `flux_balance` | training term: mean over rows of $((q(y) - \bar q)/(U L_x))^2$ |
| `boundary` | training term: inlet mismatch plus squared outlet gauge, divided by $\Delta p^2$ |

`fd` is `torch.gradient` and `fc` the spectral derivative with Fourier continuation. Precision `float32` is what training sees; `float64` uses exact float64 grids. Tables show the median over cases; the figures show every case.

In [ ]:
residuals = steady.summarize_reference_residuals(reference)
show(Markdown("**Median over cases**"))
show(steady.residual_overview(residuals).style.format(scientific))
show(Markdown("**Maximum over cases**"))
show(steady.residual_overview(residuals, statistic="max").style.format(scientific))
plots.plot_crop_sweep(residuals)
plots.plot_reference_terms(residuals, settings=(("fd", 0), ("fd", 2), ("fc", 2)));

### Momentum decomposition and the 2/3 trace part

RMS of each momentum part over the interior at the training precision. Pressure gradient and drag balance each other; the viscous Brinkman term is small, and its trace part $-\tfrac{2}{3}\nabla((\mu/\varepsilon)\nabla\cdot u)/G$ is smaller still. The COMSOL template sets Brinkman *Incompressible flow*, whose stress keeps the trace term (it vanishes only where $\nabla\cdot u = 0$); the loss uses the same stress.

In [ ]:
fields = steady.evaluate_backends(reference)
decomposition = steady.momentum_term_table({label: fields[label] for label in ("fd", "fc")}, case_labels=reference.case_labels)
show(decomposition.groupby(["backend", "crop"]).median(numeric_only=True).style.format(scientific))
plots.plot_term_decomposition(decomposition[decomposition["crop"] == DECOMPOSITION_CROP]);

### Boundary behaviour

Momentum-residual RMS per band of cell distance to the nearest edge, profiles along each edge (corners excluded), and residual maps of one case. The side walls are no-slip in COMSOL, so the stored velocity drops to zero over a Brinkman layer far thinner than one grid cell.

In [ ]:
show(steady.band_overview(steady.residual_band_table(fields, case_labels=reference.case_labels)).style.format(scientific))
plots.plot_edge_profiles({label: steady.edge_profiles(steady.residual_magnitude(fields[label])) for label in ("fd", "fc")})
plots.plot_residual_maps(
    [fields["fd"], fields["fc"]], case_index=MAP_CASE, case_label=reference.case_labels[MAP_CASE], extent=reference.extent(MAP_CASE)
);

### Finite differences against spectral derivatives

Band RMS of the derivative difference, divided by the interior RMS of the finite-difference derivative of the same field and direction (median over cases, float64).

In [ ]:
comparison = steady.compare_derivatives(reference)
show(steady.derivative_overview(comparison).style.format(scientific))
plots.plot_derivative_difference(comparison[comparison["backend"] == "fc"]);

### Sensitivity: does a term separate field errors from the reference floor?

Each perturbation is added to the reference outputs and every loss term is divided by its unperturbed reference value (ratio 1: invisible). `smooth` adds a unit-RMS low-mode cosine field ($\le 4$ half-waves per axis), `noise` iid Gaussian values, both scaled to the stated fraction of each perturbed channel's RMS; `velocity_scale` multiplies $u, v$; `pressure_offset` adds a fraction of $\Delta p$. Seeds are explicit (`seed` column); settings are the two maintained configurations plus FD with crop 2; precision is float32.

In [ ]:
sensitivity = steady.evaluate_sensitivity(reference)
for term in ("momentum", "flux_balance", "boundary"):
    show(Markdown(f"**{term}: perturbed / reference (median and minimum over cases and seeds)**"))
    show(steady.sensitivity_overview(sensitivity, term=term).style.format("{:.3g}"))
plots.plot_sensitivity(sensitivity);

## Transient drying

### Where balance terms are persisted

The Generation contract states the total water balance $\tfrac{d}{dt}(m_{w,gr} + m_{v,gas}) = \dot m_{v,in} - \dot m_{v,out}$. Its COMSOL global series live only in the source `case.h5`; the Dataset and the evaluation artifacts carry the fields, from which grain water and evaporation follow exactly up to quadrature, pore-gas vapour only with the maintained Magnus saturation pressure (COMSOL evaluates its own), and boundary fluxes only as their advective part. The stored states are hourly samples of an adaptive BDF integration; the transient template (SHA-256 `46e7112c…`) sets order 1–2 and relative tolerance 0.01, and neither setting is recorded in the case provenance. Hourly states cannot separate sub-hourly variation from solver error.

In [ ]:
show(transient.balance_term_availability())

### Balance closures per case

Closures are storage change minus integrated net source, summed over the case and divided by the water removed from the grain. Rules: trapezoidal (`_trapezoid`), left endpoint (`_left`) and, for the grain balance, Simpson. Rule differences bound only the quadrature error of smoothly varying rates; agreement between rules does not exclude sub-hourly variation, so a closure error is either sub-hourly dynamics or solver error. Field-versus-COMSOL columns give the largest relative difference over stored times. Inlet deviations compare the stored inlet row with the scheduled inlet state, separately for the interior of the inlet and its two wall corners. `negative_water_points` counts negative stored water (integrated as zero); `validity_invalid_points` counts reference values that the maintained evaluation validity rule (`T` range, $0 \le \varphi \le 1$, non-negative water) would flag in a prediction. Sorption medians are RMS residuals of the two compartment equations divided by the RMS storage rate.

In [ ]:
cases = transient.evaluate_dataset(TRANSIENT_DATASET, storage_root=STORAGE_ROOT, max_cases=MAX_CASES)
show(transient.summary_table(cases))
plots.plot_transient_balances({item.label: item.balance for item in cases})
plots.plot_field_vs_comsol({item.label: item.comparison for item in cases})
plots.plot_sorption_residuals({item.label: item.sorption for item in cases});

## Not covered here

- Energy balance (latent heat, wall loss, inlet enthalpy): no heat-flux exports.
- Diffusive boundary fluxes and the COMSOL saturation concentration: not persisted.
- Sub-hourly dynamics: only hourly states are stored.
- Grid convergence and physical model validity (e.g. Forchheimer effects at the observed velocities): outside the scope of an implementation check.